In [131]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp

In [132]:
df_network = pd.read_excel('Network_rural.xlsx')
node_list = pd.read_excel('Nodes.xlsx')
df_demand = pd.read_excel('Demand.xlsx') #demand for 27.1.2019

In [133]:
df_demand = df_demand / 50 #turining it to pu 

In [134]:
def node_index(node):
    return(node_list[node_list['node']==node].index[0])

In [135]:
def node_name(n):
    return(node_list['node'][n])

In [136]:
node_number = len(node_list)
T = len(df_demand)
partitions = 10
v_min = 0.95
v_max = 1.05
v_nom = 1
q_factor_demand = np.tan(np.arccos(0.8))
q_factor_DG = np.tan(np.arccos(0.8))
delta_s = v_nom * df_network['IMAX'] / partitions
R = range(1,partitions+1)
const = []   
t = 0

In [137]:
def load(n):
    return(np.array(df_demand.get(node_list['node'][n], default=np.zeros(df_demand.shape[0]))))

In [138]:
# power trading variables
power_act_poc = cp.Variable(shape = (node_number,T))                         # active power of poc
power_react_poc = cp.Variable(shape = (node_number,T))                       # reactive power of poc
DG_HC = cp.Variable(shape = (node_number,T), nonneg = True)                  # DG hosting capacity

# network related parameters
flow_act_pos = cp.Variable(shape = (len(df_network),T), nonneg = True)      # positive active power flow
flow_act_neg = cp.Variable(shape = (len(df_network),T), nonneg = True)      # negative active power flow
flow_react_pos = cp.Variable(shape = (len(df_network),T), nonneg = True)    # positive reactive power flow
flow_react_neg = cp.Variable(shape = (len(df_network),T), nonneg = True)    # negative reactive power flow
voltage_square = cp.Variable(shape = (node_number,T), nonneg = True)        # voltage square
current_square = cp.Variable(shape = (len(df_network),T), nonneg = True)    # current square
power_act_delta = np.zeros((partitions+1), dtype=type(cp.Variable()))
for i in R:
    power_act_delta[i] = cp.Variable(shape=(len(df_network),T), nonneg = True)
power_react_delta = np.zeros((partitions+1), dtype=type(cp.Variable()))
for i in R:
    power_react_delta[i] = cp.Variable(shape=(len(df_network),T), nonneg = True)
par1 = np.zeros((partitions+1), dtype=type(cp.Variable()))
for i in R:
    par1[i] = cp.Variable(shape=(len(df_network),T), nonneg = True)
par2 = np.zeros((partitions+1), dtype=type(cp.Variable()))
for i in R:
    par2[i] = cp.Variable(shape=(len(df_network),T), nonneg = True)

In [139]:
for t in range(T):
    for n in range(node_number):
                # balance constraints for active power
        const += [power_act_poc[n,t] # active power to the local grid
                  - (load(n)[t] )   # load of the system
                  + DG_HC[n,t]      # DG injected to the grid
                  - (np.array([df_network.fr == node_name(n)]).astype(int) @ flow_act_pos)[0,t]
                  + (np.array([df_network.fr == node_name(n)]).astype(int) @ flow_act_neg)[0,t]
                  - ((np.array([df_network.fr == node_name(n)]).astype(int) 
                  * np.array(df_network.RL)) @ current_square)[0,t]
                  + (np.array([df_network.to == node_name(n)]).astype(int) @ flow_act_pos)[0,t]
                  - (np.array([df_network.to == node_name(n)]).astype(int) @ flow_act_neg)[0,t] == 0]

In [140]:
for t in range(T):
    for n in range(node_number):
        const += [power_react_poc[n,t] # reactive power to the local grid
                  - q_factor_demand * (load(n)[t]) # reactive power consumed from demand
                  + q_factor_DG * DG_HC[n,t] # reactive power injected from DGs
                  - (np.array([df_network.fr == node_name(n)]).astype(int) @ flow_react_pos)[0,t]
                  + (np.array([df_network.fr == node_name(n)]).astype(int) @ flow_react_neg)[0,t]
                  - ((np.array([df_network.fr == node_name(n)]).astype(int) 
                  * np.array(df_network.XL)) @ current_square)[0,t]
                  + (np.array([df_network.to == node_name(n)]).astype(int) @ flow_react_pos)[0,t]
                  - (np.array([df_network.to == node_name(n)]).astype(int) @ flow_react_neg)[0,t] == 0]

In [141]:
for t in range(T):
    for n in range(node_number):
        # Voltage constraints
        const += [voltage_square[n,t] <= np.square(v_max)] #maximum voltage limit
        const += [np.square(v_min) <= voltage_square[n,t]] #minimum voltage limit
        const += [voltage_square[0,t] == np.square(v_nom)] #voltage of slack bus
        const += [DG_HC[0,t] == 0] #We don't have DG at slack bus

In [142]:
for t in range(T):
    for n in range(node_number):
# Constraints related to the poc node
        if n != 0:
            const += [power_act_poc[n,t] == 0]
            const += [power_react_poc[n,t] == 0]

In [143]:
for t in range(T):
    for net in range(len(df_network)):
        # network constraints:
        # voltage and power relationship:
        const += [voltage_square[node_index(df_network.loc[net,'fr']),t] -
        voltage_square[node_index(df_network.loc[net,'to']),t] -
        (np.square(df_network.RL[net]) + np.square(df_network.XL[net])) * current_square[net,t] -
        2 * df_network.RL[net] * (flow_act_pos[net,t] - flow_act_neg[net,t]) -
        2 * df_network.XL[net] * (flow_react_pos[net,t] - flow_react_neg[net,t]) == 0]

        # active and reactive power flow constraints
        const += [flow_act_pos[net,t] + flow_act_neg[net,t] <= v_nom * df_network.IMAX[net]]
        const += [flow_react_pos[net,t] + flow_react_neg[net,t] <= v_nom * df_network.IMAX[net]]

        # voltage constraint
        const += [current_square[net,t] <= np.square(df_network.IMAX[net])]

In [144]:
for t in range(T):
    for net in range(len(df_network)):
        # piece-wise linearization-related constraints
        const += [np.square(v_nom) * current_square[net,t] == 
        sum(par1[:])[net,t] + sum(par2[:])[net,t]]

        const += [flow_act_pos[net,t] + flow_act_neg[net,t] ==
        sum(power_act_delta[:])[net,t]]

        const += [flow_react_pos[net,t] + flow_react_neg[net,t] ==
        sum(power_react_delta[:])[net,t]]

        for r in R:
            const += [par1[r][net,t] == (2*r-1) * delta_s[net] * power_act_delta[r][net,t]]
            const += [par2[r][net,t] == (2*r-1) * delta_s[net] * power_react_delta[r][net,t]]
            const += [power_act_delta[r][net,t] <= delta_s[net]]
            const += [power_react_delta[r][net,t] <= delta_s[net]]
    

In [145]:
for t in range(T):
    const += [cp.sum(power_act_poc[:,t]) <= (800/1.31) /50]

In [146]:
for t in range(T):
    const += [cp.sum(power_react_poc[:,t]) <= (800/1.68) /50]

In [147]:
for t in range(T):
    const += [cp.sum(power_act_poc[:,t]) >= -(800/1.31) /50]

In [148]:
for t in range(T):
    const += [cp.sum(power_react_poc[:,t]) >= -(800/1.68) /50]

In [149]:
obj = cp.sum(cp.sum(DG_HC))

In [150]:
problem = cp.Problem(cp.Maximize(obj), const)

In [151]:
obj_value = problem.solve(verbose=True)

                                     CVXPY                                     
                                     v1.3.2                                    
(CVXPY) Mar 06 11:35:13 AM: Your problem has 11856 variables, 13440 constraints, and 0 parameters.
(CVXPY) Mar 06 11:35:14 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Mar 06 11:35:14 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Mar 06 11:35:14 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Mar 06 11:35:15 AM: Compiling problem (target solver=ECOS).
(CVXPY) Mar 06 11:35:15 AM: Reduction chain: FlipObjective -> Dcp2Cone -> CvxAttr2Constr 

In [152]:
obj_value * 50

12243.771761292688

In [155]:
DG = np.array(DG_HC.value * 50)

In [164]:
HC_non_slack = DG[1:,:]

In [167]:
HC_nodal = np.min(HC_non_slack, axis=1)

In [168]:
HC_nodal

array([169.54830321,  28.59989646,  20.79479701,  18.27446834,
       169.15219121,  24.31394066,  16.55142523,  13.44041565,
        11.97987025,  11.45868918])

In [170]:
sum(HC_nodal)

484.1139972099959